# Toto 2.0 — DIMER forecasting tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** zero-shot multivariate probabilistic forecasting with the pinned Toto 2.0 2.5B checkpoint

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim. No gradient training or fine-tuning occurs.

**Learning objectives:** bootstrap the exact repository revision in a fresh GPU runtime, validate deterministic sample or BYOD input, make a leakage-safe chronological holdout, compare Toto with a naive last-value baseline, inspect median/model-quantile outputs, measure tutorial error and empirical q10–q90 coverage, and export machine-readable forecasts plus provenance.


## Prerequisites

Use a CUDA GPU runtime for this 2.5B release-reference path. Optional BYOD upload is gated off by default. BYOD expects a UTF-8 CSV with a unique `timestamp` column and one or more finite numeric target columns in chronological order. Missing target values are rejected by this initial DIMER contract rather than silently imputed. Do not upload confidential or restricted data to a hosted notebook environment unless authorized. Uploaded inputs remain in the notebook runtime and are not sent to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

When no repository checkout exists, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies are directly pinned. If installation replaces an already imported core package, the cell fails with a restart instruction rather than continuing with mixed versions.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/toto-forecasting-pipeline.git'
REPO_NAME = 'toto-forecasting-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked = {'torch': 'torch', 'numpy': 'numpy', 'pandas': 'pandas'}
    # Distribution versions of core packages that are already imported, captured before
    # installation. Metadata is compared with metadata afterwards: torch.__version__ carries a
    # local build label (for example 2.6.0+cu124) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    loaded = {distribution: _installed_version(distribution) for distribution, module in tracked.items() if module in sys.modules}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy as np, pandas as pd, torch
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'A CUDA GPU is required for the Toto 2.5B release-reference path.'

## 2. Create deterministic sample or optional BYOD

The default two-variate sample is generated reproducibly. For BYOD, the raw CSV header is checked before pandas ingestion so duplicate columns cannot be silently renamed. Timestamps must parse, increase strictly, and be regularly spaced for the chronological tutorial. The production target validator checks target shape and finiteness before baseline or model execution.

In [ ]:
import csv
from toto_forecasting_pipeline.validation import validate_target

USE_BYOD = False
if USE_BYOD:
    from google.colab import files
    name = next(iter(files.upload()))
    with open(name, newline='', encoding='utf-8-sig') as handle:
        header = next(csv.reader(handle), [])
    if not header or len(header) != len(set(header)):
        raise ValueError('CSV must have non-empty unique column names; duplicate headers are rejected before pandas ingestion')
    if 'timestamp' not in header:
        raise ValueError('BYOD CSV must contain a timestamp column')
    target_columns = [column for column in header if column != 'timestamp']
    if not target_columns:
        raise ValueError('BYOD CSV must contain at least one numeric target column')
    df = pd.read_csv(name)
    timestamps = pd.to_datetime(df['timestamp'], errors='raise')
    if not timestamps.is_monotonic_increasing or timestamps.duplicated().any():
        raise ValueError('timestamps must be unique and strictly increasing')
    if len(timestamps) > 2 and timestamps.diff().dropna().nunique() != 1:
        raise ValueError('timestamps must be regularly spaced for this tutorial path')
    values = df[target_columns].to_numpy(dtype=float).T
else:
    rng = np.random.default_rng(11)
    t = np.arange(320)
    v1 = 0.01 * t + np.sin(t / 9) + rng.normal(0, 0.04, len(t))
    v2 = 0.5 * v1 + np.cos(t / 13) + rng.normal(0, 0.04, len(t))
    values = np.vstack([v1, v2])
values = validate_target(values)
print({'shape': values.shape, 'sample': 'BYOD' if USE_BYOD else 'deterministic synthetic'})

## 3. Chronological holdout and naive baseline

The final horizon is withheld from model context, enforcing chronological evaluation and preventing future-target leakage. Last-value is the naive history-only baseline. This single tutorial holdout is not a deployment benchmark or a dispersion estimate.

In [ ]:
from toto_forecasting_pipeline import TotoForecastPipeline, last_value_baseline, mae, rmse, interval_coverage, MODEL_ID, MODEL_REVISION
horizon = 48
context = values[:, :-horizon]
truth = values[:, -horizon:]
baseline = last_value_baseline(context, horizon)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'context': context.shape[1], 'horizon': horizon, 'baseline_mae': mae(truth, baseline), 'baseline_rmse': rmse(truth, baseline)})

## 4. Resolve Toto 2.0 and forecast

Toto returns q=0.1–0.9. The q=0.5 value is the median point forecast; q=0.1/q=0.9 are model quantiles, not guaranteed confidence bounds. `decode_block_size=768` is explicit for the tutorial; the public API also accepts `None` for a single forward-pass decode. Upstream consumes the context in 32-step patches, so the wrapper left-pads the 272-step tutorial context to 288 with masked (unobserved) positions, as the upstream GluonTS adapter does; the applied `context_padding` is recorded in provenance.

In [ ]:
pipe = TotoForecastPipeline.from_pretrained(device='cuda')
result = pipe.forecast(context, horizon=horizon, decode_block_size=768)
pred = result['median']
lo = result['quantiles'][:, 0, :]
hi = result['quantiles'][:, -1, :]
metrics = {'mae': mae(truth, pred), 'rmse': rmse(truth, pred), 'baseline_mae': mae(truth, baseline), 'baseline_rmse': rmse(truth, baseline), 'q10_q90_empirical_coverage': interval_coverage(truth, lo, hi)}
print(metrics)

## 5. Export forecasts and provenance

Machine-readable outputs preserve variate/time-step alignment, truth for this holdout, baseline, all quantiles, repository revision, model revision, runtime, and decode strategy.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
rows = []
for variate in range(pred.shape[0]):
    for step in range(horizon):
        row = {'variate': variate, 'step': step + 1, 'truth': float(truth[variate, step]), 'baseline': float(baseline[variate, step])}
        for index, level in enumerate(result['quantile_levels']):
            row[f'q{int(level * 100):02d}'] = float(result['quantiles'][variate, index, step])
        rows.append(row)
pd.DataFrame(rows).to_csv('outputs/toto_forecast.csv', index=False)
prov = {'repository_revision': REPO_SHA, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'metrics': metrics, 'context_length': context.shape[1], 'horizon': horizon, 'decode_block_size': result['decode_block_size'], 'context_padding': result['context_padding'], 'patch_size': result['patch_size'], 'quantile_levels': list(result['quantile_levels']), 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'device': pipe.device}, 'sample': 'BYOD' if USE_BYOD else 'deterministic synthetic'}
with open('outputs/toto_provenance.json', 'w', encoding='utf-8') as handle:
    json.dump(prov, handle, indent=2)
print(['outputs/toto_forecast.csv', 'outputs/toto_provenance.json'])

## Interpretation and limits

No gradient training or fine-tuning occurs. The tutorial's error and empirical q10–q90 coverage are tied to one synthetic chronological holdout. Model quantiles must not be described as guaranteed confidence intervals. Toto 2.0 exogenous-variable support and fine-tuning are deliberately excluded because they are not part of the current upstream 2.0 inference release.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Upstream model: https://huggingface.co/Datadog/Toto-2.0-2.5B
- Upstream code: https://github.com/DataDog/toto
- Technical report: https://arxiv.org/abs/2605.20119
